# ECE 304 Assignment:1

Name: Baibhaw Kumar

Roll No: 2310110712

Course: AI-ML



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

In [105]:
data = pd.read_csv("Dataset_Assignment_1.csv") #please change path before executing the code...

x = data[['x1','x2']].values
y = data['y'].values
print("total_samples", len(x))
data.head()

total_samples 10000


,x1,x2,y
0,-0.782278,1.143939,1
1,1.384366,0.760843,3
2,0.613042,1.818160,3
3,0.636705,-1.362102,2
4,-1.074158,0.923382,1


# Question 1: Bayesian Classifier
 
We use given means, variances, and priors to design the Bayesian classifier and calculate the probability of error.

In [ ]:
#Given parameters

The dataset consists of 10,000 samples belonging to four classes.

Each class follows a Gaussian distribution with:

- Means: (-1,-1), (-1,1), (1,-1), (1,1)
- Variance: = 0.1
- Priors: [0.1, 0.3, 0.25, 0.35]

The first 5,000 samples are used for training and the remaining for testing.

means = {
    0: np.array([-1,-1]),
    1: np.array([-1,1]),
    2: np.array([1,-1]),
    3: np.array([1,1])
}

In [9]:
priors = {
    0: 0.10,
    1: 0.3,
    2: 0.25,
    3: 0.35
}

In [14]:
variance = 0.1
sigma = np.sqrt(variance)
print("standard deviation:" , sigma)


standard deviation: 0.31622776601683794


# Gaussian Probability Density Function

The likelihood of a feature value x belonging to a class with mean μ and
standard deviation σ is modeled using a Gaussian distribution.

The probability density function is given by:

P(x | μ, σ) = (1 / (√(2π) σ)) · exp(−(x − μ)² / (2σ²))

This function computes the probability of observing x under a normal distribution
with parameters μ and σ.

It is used to calculate the class-conditional likelihood in Bayesian classification.


In [15]:
def gaussian_pdf(x,mu,sigma):

    coeff = 1/(np.sqrt(2*np.pi)*sigma)
    exp_term = np.exp(-((x-mu)**2)/(2*sigma**2))

    return  coeff * exp_term

# Accuracy Computation

Classification accuracy is defined as the proportion of correctly classified samples.

Accuracy = (Number of Correct Predictions) / (Total Number of Samples)

In this implementation, accuracy is calculated using NumPy by comparing the
true labels with the predicted labels.

The expression (y_true == y_pred) produces a Boolean array where each element
is True if the prediction is correct and False otherwise.

Taking the mean of this array converts True to 1 and False to 0, resulting in
the overall classification accuracy.


In [16]:
def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)

# True Bayesian Classifier Implementation

This function implements the optimal Bayesian classifier using the true
distribution parameters.

For each input sample x, the following steps are performed:

1. The class-conditional likelihood P(x | Cᵢ) is computed for each class
   using the Gaussian probability density function.

2. The posterior probability for each class is calculated as:

   P(Cᵢ | x) = P(x | Cᵢ) · P(Cᵢ)

   where P(Cᵢ) is the prior probability of class i.

3. The class with the maximum posterior probability is selected using
   the Maximum A Posteriori (MAP) rule.

The function returns the predicted class labels for all input samples.


In [25]:
def bayes_prediction(x,means,priors,sigma):
    prediction = []

    for x in x:
        posteriors = []
        for c in range(4):
            mu = means[c]
            p1 = gaussian_pdf(x[0], mu[0], sigma)
            p2 = gaussian_pdf(x[1],mu[1], sigma)

            likelihood = p1 * p2
            posterior = priors[c] * likelihood
            posteriors.append(posterior)
        prediction.append(np.argmax(posteriors))
    return np.array(prediction)
            

In [24]:
y_pred_q1 =bayes_prediction(x,means,priors,sigma)

acc_q1 = accuracy_score(y,y_pred_q1)
error_q1 = 1 - acc_q1

print("accuracy", acc_q1)
print("error:", error_q1)

accuracy 0.9987
error: 0.0012999999999999678


# Question 2: Bayesian Classifier Using Estimated Parameters

In this part, the parameters of the Gaussian class-conditional distributions
are estimated from the training data.

The estimated parameters include:

- Class prior 
- Class mean
- variance

These estimated values are then used to construct a Bayesian classifier,
and its performance is evaluated on the test dataset.


In [66]:
data = pd.read_csv("Dataset_Assignment_1.csv")

X_raw = data[['x1','x2']].values
y_raw = data['y'].values
print("total_samples", len(x))
data.head()

total_samples 10000


,x1,x2,y
0,-0.782278,1.143939,1
1,1.384366,0.760843,3
2,0.613042,1.818160,3
3,0.636705,-1.362102,2
4,-1.074158,0.923382,1


In [82]:
training_data_x = X_raw[:5000]
training_data_y= y_raw[:5000]


test_data_x = X_raw[5000:]
test_data_y = y_raw[5000:]


print("Training_data", len(training_data_x),len(training_data_y))
print("test_data", len(test_data_x), len(test_data_y))


Training_data 5000 5000
test_data 5000 5000


# Estimation of Priors, Means, and Variances

In Question 2, the parameters of the Gaussian distributions are estimated
from the training data using statistical methods.

#### 1. Estimation of Class Priors

The prior probability of each class Cᵢ is estimated as:

πᵢ = (Number of samples in class i) / (Total number of training samples)

This represents the relative frequency of each class in the training data.

#### 2. Estimation of Class Means

For each class Cᵢ, the mean vector is estimated using the sample average:

μᵢ = (1 / Nᵢ) ∑ xₖ ,  where xₖ ∈ Cᵢ

where Nᵢ is the number of samples belonging to class i.

The mean represents the center of each class distribution.

#### 3. Estimation of  Variance

A common variance is assumed for all classes.

The variance is estimated by averaging the within-class variances:

σ² = (1 / K) ∑ Var(x | Cᵢ)

where K is the number of classes.

This provides an estimate of the spread of the data within each class.


In [83]:
priors_est = {}

for c in range(4):
    priors_est[c] = np.mean(training_data_y == c)

priors_est


{0: 0.105, 1: 0.3016, 2: 0.2456, 3: 0.3478}

In [84]:
print("Min x1:", np.min(training_data_x[:,0]))
print("Max x1:", np.max(training_data_x[:,0]))

print("Min x2:", np.min(training_data_x[:,1]))
print("Max x2:", np.max(training_data_x[:,1]))

Min x1: -2.2132570130654123
Max x1: 2.1420091626689666
Min x2: -2.219492955288202
Max x2: 2.0843167003322653


In [85]:

print("Shape:", training_data_x.shape)


Shape: (5000, 2)


In [86]:
means_est = {}


for c in range(4):
    xc = training_data_x[training_data_y == c]
    means_est[c] = np.mean(xc, axis = 0)
means_est

{0: array([-0.98827208, -0.99596136]),
 1: array([-1.01125239,  1.01401385]),
 2: array([ 1.00934609, -0.99905888]),
 3: array([0.99238921, 1.00310249])}

In [109]:
sum_var_1 = 0
sum_var_2 = 0

for c in range(4):
    
    xc = training_data_x[training_data_y == c]
    
    sum_var_1 += np.var(xc[:,0])
    sum_var_2 += np.var(xc[:,1])
est_var_1 = sum_var_1 / 4
est_var_2 = sum_var_2 / 4

print("Estimated Vars:", est_var_1, est_var_2)


Estimated Vars: 0.10095572873166755 0.0969894591453441


In [95]:
def bayes_prediction_estimation(X,means,priors):

    preds = []
    for sample in X:
        posteriors = []

        for c in range(4):
            mu = means[c]

            sigma1 = np.sqrt(est_var_1)
            sigma2 = np.sqrt(est_var_2)
            p1 = gaussian_pdf(sample[0],mu[0],sigma1)
            p2 = gaussian_pdf(sample[1],mu[1],sigma2)

            likelihood = p1*p2
            posterior = priors[c] * likelihood
            posteriors.append(posterior)

        preds.append(np.argmax(posteriors))

    return np.array(preds)

In [96]:
print("estimated priors:", priors_est)
print("estimated means:",means_est)
print("estimated vars:", est_var_1,est_var_2)

estimated priors: {0: 0.105, 1: 0.3016, 2: 0.2456, 3: 0.3478}
estimated means: {0: array([-0.98827208, -0.99596136]), 1: array([-1.01125239,  1.01401385]), 2: array([ 1.00934609, -0.99905888]), 3: array([0.99238921, 1.00310249])}
estimated vars: 0.10095572873166755 0.0969894591453441


In [98]:
y_pred_q2 = bayes_prediction_estimation(test_data_x, means_est, priors_est)

acc_q2 = accuracy_score(test_data_y, y_pred_q2)
error_q2 = 1 - acc_q2

print("Accuracy:", acc_q2)
print("Error:", error_q2)

Accuracy: 0.9986
Error: 0.0013999999999999568


In [99]:
print("True Bayesian Error     :", error_q1)
print("Estimated Bayesian Error:", error_q2)

if error_q2 > error_q1:
    print("Estimated classifier performs worse than true classifier.")
else:
    print("Estimated classifier performs similar to true classifier.")

True Bayesian Error     : 0.0012999999999999678
Estimated Bayesian Error: 0.0013999999999999568
Estimated classifier performs worse than true classifier.


In [108]:
print("assignment is done\nThank you")

assignment is done
Thank you
